# JSON to Excel Export (Template-First)

This notebook exports one workbook per mold family from `mold_data.json`.

Rules implemented:
- Always load the latest template workbook.
- Follow template sheet/table/range names as source of truth.
- Keep null `vendorShortName` as blank.
- Ignore non-numeric mold-size keys in sizing maps.

In [1]:
# ----------------------------
# Configuration — edit here before running
# ----------------------------
JSON_PATH      = "../data/mold_data.json"
TEMPLATE_PATH  = "../docs/templates/MoldFamily_(Template).xlsx"
OUTPUT_DIR     = "../data/output"
MAX_FAMILIES   = 5   # e.g. 3 for a quick smoke-test; None = all
SKIP_EXISTING  = False  # True = skip families whose .xlsx already exists in OUTPUT_DIR

In [2]:
import json
import re
import shutil
import time
from pathlib import Path

import win32com.client as win32


def to_stripped(value):
    if value is None:
        return None
    text = str(value).strip()
    return text if text else None


def as_number_or_none(value):
    if value is None:
        return None
    try:
        return float(value)
    except Exception:
        return None


def is_numeric_size_key(value):
    if value is None:
        return False
    try:
        float(str(value).strip())
        return True
    except Exception:
        return False


INVALID_SHEET_CHARS = r"[\[\]\:\*\?\/\\]"


def safe_sheet_name(name):
    name = "Sheet" if name is None else str(name)
    name = re.sub(INVALID_SHEET_CHARS, "-", name).strip()
    if len(name) > 31:
        name = name[:31]
    return name or "Sheet"


def mold_size_to_row(size_value):
    size_f = float(size_value)
    step = (size_f - 1.0) / 0.5
    rounded = round(step)
    if abs(step - rounded) > 1e-9:
        return None
    row = 9 + int(rounded)   # row 9 = mold size 1.0 (shifted +1 for new Mold Family row)
    if row < 9 or row > 43:
        return None
    return row

In [3]:
def _normalize_table_values(values):
    if values is None:
        return []
    if not isinstance(values, tuple):
        return [[values]]
    if len(values) == 0:
        return []
    first = values[0]
    if isinstance(first, tuple):
        return [list(r) for r in values]
    return [list(values)]


def read_listobject_records(ws, table_name):
    lo = ws.ListObjects(table_name)
    rows = _normalize_table_values(lo.Range.Value)
    if not rows:
        return []
    headers = rows[0]
    out = []
    for row in rows[1:]:
        out.append({headers[i]: row[i] if i < len(row) else None for i in range(len(headers))})
    return out


def build_lookup_maps(ws_master):
    vendor_rows = read_listobject_records(ws_master, "_dimVendor")
    factory_rows = read_listobject_records(ws_master, "_dimFactory")

    vendor_short_by_id = {}
    for row in vendor_rows:
        vendor_id = to_stripped(row.get("Vendor ID"))
        vendor_short = to_stripped(row.get("Vendor Short Name"))
        if vendor_id and vendor_short:
            vendor_short_by_id[vendor_id] = vendor_short

    factory_name_by_number = {}
    for row in factory_rows:
        number = row.get("Factory Number")
        name = to_stripped(row.get("Factory Name"))
        if number is None or not name:
            continue
        try:
            key = str(int(float(number)))
        except Exception:
            key = str(number).strip()
        factory_name_by_number[key] = name

    return vendor_short_by_id, factory_name_by_number


def iter_component_records(fam):
    components = fam.get("components") or {}
    for sole_type, comp_dict in components.items():
        for comp_code, comp in (comp_dict or {}).items():
            molds = comp.get("molds") or []
            for mold in molds:
                yield sole_type, comp_code, comp, mold


def collect_summary_rows(fam):
    rows = []
    for _sole_type, comp_code, _comp, mold in iter_component_records(fam):
        asset = mold.get("asset") or {}
        cap = mold.get("capacity") or {}

        comment = to_stripped(mold.get("comments"))
        remark = to_stripped(mold.get("remark"))
        if comment and remark and comment != remark:
            comment_out = f"{comment} | {remark}"
        else:
            comment_out = comment or remark

        rows.append({
            "component_code": comp_code,
            "vendor_short_name": to_stripped(mold.get("vendorShortName")) or "",
            "ownership": to_stripped(asset.get("ownership")),
            "condition": to_stripped(asset.get("condition")),
            "mold_shop": to_stripped(mold.get("moldShopName")) or to_stripped(mold.get("moldShopCode")),
            "init_season": to_stripped(mold.get("initSeason")),
            "daily_output": as_number_or_none(cap.get("dailyOutput")),
            "mold_init_cost": as_number_or_none(asset.get("moldCost")),
            "comments": comment_out,
        })
    return rows

In [4]:
def write_summary(ws_summary, family_code, fam, warnings):
    input_start_row = 7
    input_end_row = 31

    # A1 has label "Mold Family:" in template; write value to B1
    ws_summary.Cells(1, 2).Value = family_code

    brands = fam.get("brands") or []
    style_brands = [
        to_stripped(s.get("brand"))
        for s in (fam.get("stylesUsingThisFamily") or [])
        if isinstance(s, dict)
    ]
    all_brands = [b for b in brands if to_stripped(b)] + [b for b in style_brands if b]
    unique_brands = list(dict.fromkeys(b for b in all_brands if b))
    # A2 has label "Brand:" in template; write value to B2
    ws_summary.Cells(2, 2).Value = ", ".join(unique_brands) if unique_brands else None

    # Clear only the columns Python owns (avoids touching formula columns B, C, E, F, G)
    ws_summary.Range(f"A{input_start_row}:A{input_end_row}").ClearContents()
    ws_summary.Range(f"D{input_start_row}:D{input_end_row}").ClearContents()
    ws_summary.Range(f"H{input_start_row}:N{input_end_row}").ClearContents()

    rows = collect_summary_rows(fam)
    capacity = input_end_row - input_start_row + 1
    if len(rows) > capacity:
        warnings.append(f"{family_code}: Summary rows truncated from {len(rows)} to {capacity}.")
        rows = rows[:capacity]

    if not rows:
        return

    n = len(rows)
    end_row = input_start_row + n - 1

    # Batch write each column group in one COM call instead of one call per cell
    ws_summary.Range(f"A{input_start_row}:A{end_row}").Value = \
        tuple((r["component_code"],) for r in rows)
    ws_summary.Range(f"D{input_start_row}:D{end_row}").Value = \
        tuple((r["vendor_short_name"] or None,) for r in rows)
    ws_summary.Range(f"H{input_start_row}:N{end_row}").Value = \
        tuple((
            r["ownership"], r["condition"], r["mold_shop"],
            r["init_season"], r["daily_output"], r["mold_init_cost"], r["comments"],
        ) for r in rows)


def write_sourcing_rules(ws_src, fam, vendor_short_by_id, factory_name_by_number, warnings, family_code):
    start_row = 5
    end_row = 29
    capacity = end_row - start_row + 1

    # Clear only columns Python owns
    ws_src.Range(f"A{start_row}:A{end_row}").ClearContents()
    ws_src.Range(f"D{start_row}:D{end_row}").ClearContents()
    ws_src.Range(f"F{start_row}:F{end_row}").ClearContents()
    ws_src.Range(f"J{start_row}:J{end_row}").ClearContents()

    ws_src.Cells(2, 2).Value = family_code

    rules = fam.get("sourcingRules") or []
    if len(rules) > capacity:
        warnings.append(f"{family_code}: Sourcing Rules truncated from {len(rules)} to {capacity}.")
        rules = rules[:capacity]

    if rules:
        factory_vals, sole_vals, vendor_vals = [], [], []
        for rule in rules:
            factory_number = rule.get("factoryNumber")
            if factory_number is None:
                factory_name = None
            else:
                try:
                    factory_key = str(int(float(factory_number)))
                except Exception:
                    factory_key = str(factory_number).strip()
                factory_name = factory_name_by_number.get(factory_key)

            vendor_id = to_stripped(rule.get("vendorId"))
            vendor_short = vendor_short_by_id.get(vendor_id) if vendor_id else None

            factory_vals.append((factory_name,))
            sole_vals.append((to_stripped(rule.get("solePart")),))
            vendor_vals.append((vendor_short,))

        n = len(rules)
        ws_src.Range(f"A{start_row}:A{start_row + n - 1}").Value = tuple(factory_vals)
        ws_src.Range(f"D{start_row}:D{start_row + n - 1}").Value = tuple(sole_vals)
        ws_src.Range(f"F{start_row}:F{start_row + n - 1}").Value = tuple(vendor_vals)

    style_names = []
    for s in (fam.get("stylesUsingThisFamily") or []):
        if isinstance(s, dict):
            style_name = to_stripped(s.get("styleName")) or to_stripped(s.get("styleCode"))
        else:
            style_name = to_stripped(s)
        if style_name and style_name not in style_names:
            style_names.append(style_name)

    if len(style_names) > capacity:
        warnings.append(f"{family_code}: Style list truncated from {len(style_names)} to {capacity}.")
        style_names = style_names[:capacity]

    if style_names:
        ws_src.Range(f"J{start_row}:J{start_row + len(style_names) - 1}").Value = \
            tuple((name,) for name in style_names)


def write_component_sheet(ws_inv, family_code, sole_type, comp_code, comp, sizing_block, warnings):
    # A1–A5 have labels in template; write values to B1–B5
    
    ws_inv.Cells(2, 2).Value = family_code
    ws_inv.Cells(3, 2).Value = sole_type
    ws_inv.Cells(4, 2).Value = to_stripped(comp.get("displayName")) or None
    ws_inv.Cells(5, 2).Value = to_stripped(comp.get("compound")) or None

    # Clear data area (rows 9–43: mold size 1.0–18.0 + shoe sizes col)
    ws_inv.Range("B9:H43").ClearContents()

    molds = comp.get("molds") or []
    vendor_order = list(dict.fromkeys(
        to_stripped(m.get("vendorShortName")) or "" for m in molds
    ))

    max_vendor_cols = 6
    if len(vendor_order) > max_vendor_cols:
        warnings.append(
            f"{family_code}/{comp_code}: Vendor columns truncated from {len(vendor_order)} to {max_vendor_cols}."
        )
        vendor_order = vendor_order[:max_vendor_cols]

    vendor_to_col = {}
    header_row = [None] * max_vendor_cols
    for i, value in enumerate(vendor_order):
        vendor_to_col[value] = 3 + i  # C=3
        header_row[i] = value if value else None

    # Write all 6 vendor headers in one COM call (row 8)
    ws_inv.Range("C8:H8").Value = (tuple(header_row),)

    sizing_map = (sizing_block or {}).get("moldSizeToShoeSizes") or {}
    for mold_size, shoe_sizes in sizing_map.items():
        if not is_numeric_size_key(mold_size):
            continue
        row = mold_size_to_row(float(str(mold_size).strip()))
        if row is None:
            continue
        if shoe_sizes is None:
            ws_inv.Cells(row, 2).Value = None
        elif isinstance(shoe_sizes, list):
            ws_inv.Cells(row, 2).Value = ", ".join([str(x) for x in shoe_sizes])
        else:
            ws_inv.Cells(row, 2).Value = str(shoe_sizes)

    for mold in molds:
        vendor_short = to_stripped(mold.get("vendorShortName")) or ""
        if vendor_short not in vendor_to_col:
            continue
        target_col = vendor_to_col[vendor_short]

        qty_map = ((mold.get("inventory") or {}).get("qtyByMoldSize")) or {}
        for mold_size, qty in qty_map.items():
            if not is_numeric_size_key(mold_size):
                continue
            row = mold_size_to_row(float(str(mold_size).strip()))
            if row is None:
                continue
            qty_num = as_number_or_none(qty)
            if qty_num is None:
                continue
            current = as_number_or_none(ws_inv.Cells(row, target_col).Value) or 0.0
            ws_inv.Cells(row, target_col).Value = current + qty_num

In [ ]:
def export_one_family(app, template_abs, family_code, fam, out_path,
                      vendor_short_by_id, factory_name_by_number):
    warnings = []

    wb = app.Workbooks.Open(template_abs)
    try:
        ws_summary      = wb.Worksheets("Summary")
        ws_inv_template = wb.Worksheets("MoldInv_{Component}")
        ws_src          = wb.Worksheets("Sourcing Rules")

        write_summary(ws_summary, family_code, fam, warnings)
        write_sourcing_rules(ws_src, fam, vendor_short_by_id, factory_name_by_number,
                             warnings, family_code)

        components  = fam.get("components") or {}
        sizing_all  = fam.get("sizingRulesByComponent") or {}
        sheet_names = {s.Name for s in wb.Worksheets}
        created     = set()

        for sole_type, comp_dict in components.items():
            for comp_code, comp in (comp_dict or {}).items():
                ws_inv_template.Copy(After=wb.Worksheets(wb.Worksheets.Count))
                new_ws = wb.ActiveSheet  # Excel always activates the new copy after Copy()

                target = safe_sheet_name(f"MoldInv_{comp_code}")
                base   = target
                suffix = 1
                while target in sheet_names or target in created:
                    suffix += 1
                    target = safe_sheet_name(f"{base}_{suffix}")
                new_ws.Name = target
                created.add(target)
                sheet_names.add(target)

                write_component_sheet(new_ws, family_code, sole_type, comp_code, comp,
                                      sizing_all.get(comp_code) or {}, warnings)

        app.DisplayAlerts = False
        ws_inv_template.Delete()
        app.DisplayAlerts = True

        wb.SaveAs(str(out_path), FileFormat=51)
    finally:
        wb.Close(SaveChanges=False)

    return warnings


def export_all_families(json_path, template_path, output_dir,
                        max_families=None, skip_existing=False):
    data     = json.loads(Path(json_path).read_text(encoding="utf-8"))
    families = data.get("families") or {}
    if not families:
        raise ValueError("No families found in JSON.")

    out_dir      = Path(output_dir).resolve()
    template_abs = str(Path(template_path).resolve())
    out_dir.mkdir(parents=True, exist_ok=True)

    app = win32.DispatchEx("Excel.Application")
    app.Visible       = False
    app.DisplayAlerts = False

    written_files = []
    skipped_files = []
    all_warnings  = []
    total = sum(1 for i, _ in enumerate(families, 1)
                if max_families is None or i <= max_families)

    try:
        # Open template once to build lookup maps, then close — not reopened until per-family loop
        template_wb = app.Workbooks.Open(template_abs)
        vendor_short_by_id, factory_name_by_number = build_lookup_maps(
            template_wb.Worksheets("_Master_Ref")
        )
        template_wb.Close(SaveChanges=False)

        for i, (family_code, fam) in enumerate(families.items(), start=1):
            if max_families is not None and i > max_families:
                break

            out_path = out_dir / f"{family_code}.xlsx"

            if skip_existing and out_path.exists():
                skipped_files.append(str(out_path))
                print(f"[{i}/{total}] SKIP {family_code}   ", end="\r", flush=True)
                continue

            print(f"[{i}/{total}] {family_code}   ", end="\r", flush=True)

            warnings = export_one_family(
                app=app,
                template_abs=template_abs,
                family_code=family_code,
                fam=fam,
                out_path=out_path,
                vendor_short_by_id=vendor_short_by_id,
                factory_name_by_number=factory_name_by_number,
            )
            written_files.append(str(out_path))
            all_warnings.extend(warnings)

        print()  # end progress line

    finally:
        app.Quit()

    return written_files, skipped_files, all_warnings


In [6]:
t0 = time.time()

files, skipped, warnings = export_all_families(
    json_path=JSON_PATH,
    template_path=TEMPLATE_PATH,
    output_dir=OUTPUT_DIR,
    max_families=MAX_FAMILIES,
    skip_existing=SKIP_EXISTING,
)

elapsed = time.time() - t0
print(f"Exported {len(files)} workbook(s) in {elapsed:.1f}s  |  Skipped {len(skipped)}  |  Warnings {len(warnings)}")
print(f"Output: {OUTPUT_DIR}")
for path in files[:10]:
    print(" -", path)
if len(files) > 10:
    print(f" ... and {len(files) - 10} more")
for w in warnings[:30]:
    print(" !", w)
if len(warnings) > 30:
    print(f" ... and {len(warnings) - 30} more")

[5/5] SA-2255     
Exported 5 workbook(s) in 7.4s  |  Skipped 0  |  Warnings 0
Output: ../data/output
 - D:\Project\osms-mold-master-data\data\output\SA-1318.xlsx
 - D:\Project\osms-mold-master-data\data\output\SA-2017.xlsx
 - D:\Project\osms-mold-master-data\data\output\SA-2017-E.xlsx
 - D:\Project\osms-mold-master-data\data\output\SA-2253.xlsx
 - D:\Project\osms-mold-master-data\data\output\SA-2255.xlsx
